# 1&rpar; Spark



## 1.1 What is Spark?

Apache Spark is an open-source, distributed processing system uses for big data workloads. It utilizes in-memory caching and optimized query execution for fast analytic queries against data of any size. It provides developement APIs in Java, Scala, Python and R, and supports code reuse across multiple workloads, batch processing, interactive queries, real-time analytics, machine learning and graph processing.

##1.2 Why is it a popular framework?
Spark has become a game-chngeer in big data analytics due to its speed, ease of use (due to its support for Python, Scala, Java and R), scalability abd versatility as a one-stop solution for various data tasks.


Following is the breakdown on waht makes spark so special:

1. Resilient Distributed Datasets (RDDs):
They are fault tolentant, immutable, disrtibuted collection of objects that allows for efficient parallel processing.

2. In-memory Computing:
Instead of constanlh hitting the disk, Spark stores data in RAM, which reduces the time spend on disk I/O, making it massively faster.

3. Lazy Evaluation & DAGs:
Spark doesn;t execute operations right away. Instead it creates a Directed Acyclic graph (DAG) of transformation, optimizing the whole workflow before it even starts.

## 1.3 What is Spark SQL and why does it exist?



*   SparK SQl brings native support for SQL to Spark and streamlines the process of querying data stored both in RDDs and in external sources. Unlike older framewords like Hadoop MapReducem which writes intermediate data back tp physical hard drives after every step, Spark keeps data in RAM. This makes it upto 100x faster for certain workloads.


*   Before Spark SQL, big data processing often required complex procedural languages like Java or Scala to write MapReduce jobs. Spark SQL allows analysts and data scientists to use standard SQL, a language they already know to query massive datasets. It provides a specialized module within the Apche Spark framework specifically for structured data processing.



## 1.4 What is a Spark DataFrame, and why is it useful?


A Spark DataFrame is a distributed collection of data organized into named columns, conceptually identical to a table in a relational database or Pandas DataFrame. It is useful because it abstracts away the complexity of distributed computing. We interact with it as if it were a single local table, but Spark automatically handles partioning that data and executing operations across thousands of machines in the background.

# 2&rpar; Setup / Data Collection

Kaggle Dataset Link : https://www.kaggle.com/datasets/nilesh2042/airport-traffic-dataset/data


In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("nilesh2042/airport-traffic-dataset")

print("Path to dataset files:", path)

file_path = os.path.join(path, "airport_traffic_2025.csv")


Using Colab cache for faster access to the 'airport-traffic-dataset' dataset.
Path to dataset files: /kaggle/input/airport-traffic-dataset


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession \
        .builder \
        .appName("Airport Traffic") \
        .getOrCreate()

print("✅ Spark session initialized")

df = spark.read.csv(file_path, header = True, inferSchema = True)
print("Airport dataset loaded into Spark dataframe")


✅ Spark session initialized
Airport dataset loaded into Spark dataframe


# 3&rpar; Data Cleaning



## 3.1. Check for missing values

In [ ]:
from pyspark.sql.functions import isnull, when, count, col
df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+-------------+-------------+-------------+
|YEAR|MONTH_NUM|MONTH_MON|FLT_DATE|APT_ICAO|APT_NAME|STATE_NAME|FLT_DEP_1|FLT_ARR_1|FLT_TOT_1|FLT_DEP_IFR_2|FLT_ARR_IFR_2|FLT_TOT_IFR_2|
+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+-------------+-------------+-------------+
|   0|        0|        0|       0|       0|       0|         0|        0|        0|        0|        81892|        81892|        81892|
+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+-------------+-------------+-------------+



There are missing values in the column:


*   The columns FLT_DEP_IFR_2, FLT_ARR_IFR_2 and FLT_TOT_IFR_2| has got 81892 missing values, while looking at it, there is no data in these. As these columns have thousands of missing values, the best practice is to drop the column entirely, rather than dropping the rows, because those columns are too incomplete to be useful for Machine Learning anyway. And cleaning those columns would just bring in bias.

In [ ]:
cols_to_drop = ["FLT_TOT_IFR_2","FLT_ARR_IFR_2","FLT_DEP_IFR_2"]
df_clean = df.drop(*cols_to_drop)
df_clean.head()

Row(YEAR=2025, MONTH_NUM=1, MONTH_MON='JAN', FLT_DATE=datetime.date(2025, 1, 1), APT_ICAO='LATI', APT_NAME='Tirana', STATE_NAME='Albania', FLT_DEP_1=64, FLT_ARR_1=62, FLT_TOT_1=126)

## 3.2. Check for duplicates

In [ ]:
df_clean.groupBy(df_clean.columns).agg(count("*").alias('duplicates')).filter(col('duplicates')>1).show()

+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+----------+
|YEAR|MONTH_NUM|MONTH_MON|FLT_DATE|APT_ICAO|APT_NAME|STATE_NAME|FLT_DEP_1|FLT_ARR_1|FLT_TOT_1|duplicates|
+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+----------+
+----+---------+---------+--------+--------+--------+----------+---------+---------+---------+----------+



There are no duplicates in this dataset.

## 3.3. Check for outliers

In [ ]:
from pyspark.sql.functions import col, mean, stddev, abs, max, min
import pyspark.sql.types as ty

numeric_col = [f.name for f in df_clean.schema.fields if isinstance(f.dataType,(ty.IntegerType, ty.DoubleType,ty.LongType,ty.FloatType))]

print(f"checking for outliers in :{numeric_col}\n")


for column in numeric_col:

  stats = df_clean.agg(
      mean(column).alias('mean'),
      stddev(column).alias('stddev'),
      max(column).alias('max'),
      min(column).alias('min')).collect()[0]

  mean_val = stats['mean']
  std_val = stats['stddev']
  max_val = stats['max']
  min_val = stats['min']


  if std_val is None or std_val == 0:
    continue

  df_zscored = df_clean.withColumn('z_score', (col(column)-mean_val)/std_val)
  print(f"   Max: {max_val:,.2f} | Min: {min_val:,.2f} | Mean: {mean_val:,.2f} | StdDev: {std_val:,.2f}")
  outliers_df = df_zscored.filter(abs(col('z_score')) > 3)
  outliers_count = outliers_df.count()

  print(f"\n Column: '{column}'")

  if outliers_count > 0:
    print("Outliers found",{outliers_count})
  else:
    print("no Outliers")

checking for outliers in :['YEAR', 'MONTH_NUM', 'FLT_DEP_1', 'FLT_ARR_1', 'FLT_TOT_1']

   Max: 12.00 | Min: 1.00 | Mean: 6.54 | StdDev: 3.44

 Column: 'MONTH_NUM'
no Outliers
   Max: 846.00 | Min: 0.00 | Mean: 74.04 | StdDev: 125.07

 Column: 'FLT_DEP_1'
Outliers found {3248}
   Max: 844.00 | Min: 0.00 | Mean: 74.10 | StdDev: 125.08

 Column: 'FLT_ARR_1'
Outliers found {3262}
   Max: 1,688.00 | Min: 0.00 | Mean: 148.15 | StdDev: 250.12

 Column: 'FLT_TOT_1'
Outliers found {3260}


Using a threshold of > 3, i detected approximately 3260 outliers in the flight traffic columns. Because aviation traffic is heavily dominated by a few major global hub airports, these high numbers (e.g., a maximum of 1,688 total flights) represent mathematically accurate, real-world mega-hubs rather than data-entry errors. Therefore, I chose not to drop these outliers. Removing them would destroy the most important data points in the dataset and severely cripple our Machine Learning model's ability to predict high-volume traffic.

## 3.4. Check for inconsistent types

In [ ]:
import pyspark.sql.types


print("--- Current Schema ---")
df_clean.printSchema()

string_columns = [f.name for f in df.schema.fields if isinstance(f.dataType, pyspark.sql.types.StringType)]
print(f"\nString Columns to review: {string_columns}")

df_clean.select(string_columns).show(20, truncate=False)

--- Current Schema ---
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH_NUM: integer (nullable = true)
 |-- MONTH_MON: string (nullable = true)
 |-- FLT_DATE: date (nullable = true)
 |-- APT_ICAO: string (nullable = true)
 |-- APT_NAME: string (nullable = true)
 |-- STATE_NAME: string (nullable = true)
 |-- FLT_DEP_1: integer (nullable = true)
 |-- FLT_ARR_1: integer (nullable = true)
 |-- FLT_TOT_1: integer (nullable = true)


String Columns to review: ['MONTH_MON', 'APT_ICAO', 'APT_NAME', 'STATE_NAME']
+---------+--------+--------------------+----------------------+
|MONTH_MON|APT_ICAO|APT_NAME            |STATE_NAME            |
+---------+--------+--------------------+----------------------+
|JAN      |LATI    |Tirana              |Albania               |
|JAN      |UDYZ    |Yerevan             |Armenia               |
|JAN      |LOWG    |Graz                |Austria               |
|JAN      |LOWI    |Innsbruck           |Austria               |
|JAN      |LOWK    |Klagenfurt 

To verify that the types are consistent, we look at the schema and the string columns. Since there are no numbers 'in disguise' in any of these columns, it is safe to say that there are no inconsistencies.

## 3.5. Check for problematic mistypings or misspellings


In [ ]:
from pyspark.sql.functions import col

print("--- Unique Months ---")
df_clean.groupBy("MONTH_MON").count().orderBy("MONTH_MON").show()

print("--- State Name Groupings ---")
df_clean.groupBy("STATE_NAME").count().orderBy("STATE_NAME").show(50, truncate=False)

--- Unique Months ---
+---------+-----+
|MONTH_MON|count|
+---------+-----+
|      APR| 9563|
|      AUG| 9879|
|      DEC| 9821|
|      FEB| 8808|
|      JAN| 9658|
|      JUL| 9919|
|      JUN| 9594|
|      MAR| 9794|
|      MAY| 9901|
|      NOV| 9564|
|      OCT| 9892|
|      SEP| 9627|
+---------+-----+

--- State Name Groupings ---
+---------------------------+-----+
|STATE_NAME                 |count|
+---------------------------+-----+
|Albania                    |365  |
|Armenia                    |365  |
|Austria                    |2190 |
|Belgium                    |1824 |
|Bosnia and Herzegovina     |365  |
|Bulgaria                   |365  |
|Croatia                    |457  |
|Cyprus                     |730  |
|Czech Republic             |1429 |
|Denmark                    |365  |
|Estonia                    |723  |
|Finland                    |365  |
|France                     |22205|
|Georgia                    |1095 |
|Germany                    |5469 |
|Greece     


I hunted for misspellings in the categorical columns by using `groupBy().count().orderBy()` to aggregate the unique text values alphabetically. I specifically checked the MONTH_MON column to ensure only valid 3-letter month abbreviations existed, and the STATE_NAME column to ensure geographic locations were standardized without duplicate typos (e.g., ensuring there wasn't both a "Texas" and a "Texs").

# 4&rpar; Data Exploration

## 4.1. Schema

In [ ]:
df_clean.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH_NUM: integer (nullable = true)
 |-- MONTH_MON: string (nullable = true)
 |-- FLT_DATE: date (nullable = true)
 |-- APT_ICAO: string (nullable = true)
 |-- APT_NAME: string (nullable = true)
 |-- STATE_NAME: string (nullable = true)
 |-- FLT_DEP_1: integer (nullable = true)
 |-- FLT_ARR_1: integer (nullable = true)
 |-- FLT_TOT_1: integer (nullable = true)



## 4.2. Shape of dataset

In [ ]:
print((df_clean.count(), len(df_clean.columns)))

(116020, 10)


## 4.3. Summary statistics of dataset

In [ ]:
df_clean.describe().show()

+-------+------+------------------+---------+--------+--------+--------------+------------------+------------------+-----------------+
|summary|  YEAR|         MONTH_NUM|MONTH_MON|APT_ICAO|APT_NAME|    STATE_NAME|         FLT_DEP_1|         FLT_ARR_1|        FLT_TOT_1|
+-------+------+------------------+---------+--------+--------+--------------+------------------+------------------+-----------------+
|  count|116020|            116020|   116020|  116020|  116020|        116020|            116020|            116020|           116020|
|   mean|2025.0| 6.542501292880538|     NULL|    NULL|    NULL|          NULL| 74.04344940527496| 74.10440441303224|148.1478538183072|
| stddev|   0.0|3.4382644160007816|     NULL|    NULL|    NULL|          NULL|125.07416443254758|125.07532528531722|250.1230172639208|
|    min|  2025|                 1|      APR|    BIKF|    Abad|       Albania|                 0|                 0|                0|
|    max|  2025|                12|      SEP|    UGTB| 

## 4.4. Dataset Schema & Feature Descriptions

Below is the data dictionary detailing the core geographical and temporal traffic features utilized in this dataset:

| Column Name | Description | Data Format / Range |
| :--- | :--- | :--- |
| **`MONTH_NUM`** | The numerical value of the month. | Integer (1 - 12) |
| **`MONTH_MON`** | The 3-letter abbreviation of the month. | String (e.g., JAN, FEB, MAR) |
| **`FLT_DATE`** | The specific date of the traffic measurement. | Date (YYYY-MM-DD) |
| **`APT_ICAO`** | The 4-letter International Civil Aviation Organization airport code. | String (e.g., LATI, UDYZ) |
| **`APT_NAME`** | The official name of the airport. | String |
| **`STATE_NAME`** | The country or administrative state where the airport is located. | String |
| **`FLT_DEP_1`** | Total number of flight departures for the day. | Integer |
| **`FLT_ARR_1`** | Total number of flight arrivals for the day. | Integer |
| **`FLT_TOT_1`** | Total daily flight movements (Arrivals + Departures). | Integer |



## 4.5. Distribution of the data

In [ ]:
from pyspark.sql.functions import avg, round

print("--- 1. Distribution of total flights (FLT_TOT_1)---")
df_clean.select("FLT_TOT_1").summary("count","min","25%","50%","75%","max").show()


--- 1. Distribution of total flights (FLT_TOT_1)---
+-------+---------+
|summary|FLT_TOT_1|
+-------+---------+
|  count|   116020|
|    min|        0|
|    25%|       12|
|    50%|       39|
|    75%|      174|
|    max|     1688|
+-------+---------+



The percentile summary revelas that the distirbution of daily flight traffic is heavily right skewed with a very long tail.

*   The median (50 percentile) is only 39 flights, and 75% of the data falls below 174 flights per day. This indicates that the vast majority of records in this dataset belong to a smaller, regional airports with relatively low daiy traffic.
*   The maximum is 1688 flights per day indicating the presence of a "global mega-hub".

Understanding this right-skewed distribution is critical for our Machine Learning strategy. It confirms that "average" (mean) traffic is heavily distorted by the mega-hubs. It also validates our earlier decision to retain the statistical outliers. if we had dropped the high-volume days, we would have effectively blinded our regression model to how major international airports operate.



##4.6. Seasonality in the dataset

In [ ]:
print("--- Seasonality (Average flights per month)---")

seasonality_df = df_clean.groupBy("MONTH_NUM") \
                 .agg(round(avg("FLT_TOT_1"),2).alias("Avg_Daily_flights")) \
                 .orderBy("MONTH_NUM")
seasonality_df.show()

--- Seasonality (Average flights per month)---
+---------+-----------------+
|MONTH_NUM|Avg_Daily_flights|
+---------+-----------------+
|        1|           118.77|
|        2|           125.31|
|        3|           130.93|
|        4|           149.46|
|        5|           157.05|
|        6|           166.56|
|        7|           169.69|
|        8|           170.03|
|        9|           167.46|
|       10|           158.73|
|       11|           130.68|
|       12|           130.11|
+---------+-----------------+



By grouping the data by MONTH_NUM and calculating the average daily flights, the results demonstrate a clear and strong seasonal trend in global aviation traffic.


*   Traffic volume is at its lowest in the post-holiday winter months, bottoming out in January with an average of 118 daily flights.
*  Volume then steadily climbs throughout the spring and reaches its peak during the summer travel season, specifically July and August, hitting an average of roughly 170 daily flights.

*    As fall approaches, traffic steadily declines back toward the winter baseline.


Why is this information useful?

This validates our domain knowlegde that global travel demand is heavily dicated by summer holidays and vacation seasons. Airlines and airports must scale their operations like staffing, fueling and logistics significantly during these months to handle this ~43% increase in average daily traffic.





#5 &rpar; Machine learning


## 5.1. What ML task and algorithm have you chosen and why?
I choose to do a Supervised Regression task using RandomForestRegressor. The goal is to predict FLT_TOT_1, the total daily flights.

I specifically used Random Forest beacuse the flight traffic follows a non-linear seasonal curve, which can be handled well by decision trees.

The features I will be using are MONTH_NUM, STATE_NAME and APT_ICAO. I expliclty excluded FLT_DEP_1 and FLT_ARV_1 to prevent data leakage to the model.

## 5.2. Explain what the algorithm will do with your data:

The algorithm analyzes the training data to find patterns between the time of year (MONTH_NUM), geographic location (STATE_NAME) and specific airport (APT_ICAO) and the resulting total flights. The decision tress routes the data towards a final mathematical estimation of how many flights that specific location handles on that day.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

indexer_state = StringIndexer(inputCol = "STATE_NAME", outputCol = "State_index")
df_step_1 = indexer_state.fit(df_clean).transform(df_clean)

indexer_apt = StringIndexer(inputCol = "APT_ICAO", outputCol = "Apt_index")
df_indexed = indexer_apt.fit(df_step_1).transform(df_step_1)

assembler = VectorAssembler(inputCols=["MONTH_NUM",'State_index',"Apt_index"] , outputCol = "features")
data = assembler.transform(df_indexed)

data =  data.withColumnRenamed("FLT_TOT_1","label")

train_data, test_data = data.randomSplit([0.8 , 0.2], seed = 42)

rf = RandomForestRegressor(labelCol = "label", featuresCol = "features", numTrees = 30 , maxBins = 500,  seed = 42)
model = rf.fit(train_data)
print("Training random forest regressor model")

predictions = model.transform(test_data)
print("\n--- Sample predictions (prediced vs actual flights) ---")
predictions.select("MONTH_NUM","STATE_NAME","APT_ICAO","label","prediction").show(5, truncate = False)


rmse_evaluator = RegressionEvaluator(labelCol= "label", predictionCol= "prediction", metricName = "rmse")
rmse = rmse_evaluator.evaluate(predictions)


r2_evaluator = RegressionEvaluator(labelCol= "label", predictionCol= "prediction", metricName = "r2")
r2 = r2_evaluator.evaluate(predictions)

print(f" Model RMSE: {rmse:,.2f} flights\n")
print(f" Model R- squared {r2:.4f}")


Training random forest regressor model

--- Sample predictions (prediced vs actual flights) ---
+---------+----------+--------+-----+------------------+
|MONTH_NUM|STATE_NAME|APT_ICAO|label|prediction        |
+---------+----------+--------+-----+------------------+
|1        |Belgium   |EBBR    |419  |422.49901614152225|
|1        |Germany   |EDDB    |423  |450.8165562396111 |
|1        |Germany   |EDDE    |2    |89.46394242274862 |
|1        |Germany   |EDDL    |251  |289.4255407537391 |
|1        |Germany   |EDDV    |76   |118.77511384536095|
+---------+----------+--------+-----+------------------+
only showing top 5 rows
 Model RMSE: 73.13 flights

 Model R- squared 0.9162


## 5.3 How did you do and what do your chosen metrics mean?

I evaluated the model's success on the 20% unseen test data using the two metrics: **R-squared and RMSE.**




*   R-squared - The his metric indicates the proportion of the variance in traffic volume that is predictable from my features. Scoring **91.62%** means the algorithm is highly successful at capturing the underlying patterns of global airport traffic, proving that proving that seasonality and specific airport location are the true driving factors of aviation volume.



*   RMSE - This measures the average magnitude of my prediction error in actual flight units. Considering the dataset includes mega-hubs handling up to 1,688 flights a day, an average error margin of only **~73** flights is an incredibly strong and reliable outcome.





# 6&rpar; GRAPHS IN SPARK

In [ ]:
! pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.4 MB/s eta 0:00:00


##6.1. Installing Neo4j

In [ ]:
! pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 6.3 MB/s eta 0:00:00


In [ ]:
!pip install pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=a8c3cb747cc4fac4b7160257b92ee56010caf3685137d7a41eba0dedf24a7040
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [ ]:
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q graphframes neo4j

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

##6.2. Connect to AuraDB

In [ ]:
from pyspark.sql import SparkSession
from google.colab import userdata

password = userdata.get('neo_pwd').strip()
username = userdata.get('neo_user_name').strip()
uri = userdata.get('neo_uri').strip()

test_query = """
MATCH (n)
RETURN id(n) AS id, labels(n)[0] AS label, properties(n) AS props
"""

packages = [
    "org.neo4j:neo4j-connector-apache-spark_2.12:5.4.1_for_spark_3",
    "graphframes:graphframes:0.8.4-spark3.5-s_2.12"
]

try:
    spark.stop()
except:
    pass


spark = SparkSession.builder \
    .appName("HW5 Spark - GraphFrames") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.jars.repositories",
            "https://repos.spark-packages.org/") \
    .getOrCreate()


df = spark.read \
    .format("org.neo4j.spark.DataSource") \
    .option("url", uri) \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", username) \
    .option("authentication.basic.password", password) \
    .option("query", test_query) \
    .load()

df.show(5)

+---+--------+--------------------+
| id|   label|               props|
+---+--------+--------------------+
|  0|Customer|{loyalty_tier -> ...|
|  1|Customer|{loyalty_tier -> ...|
|  2|Customer|{loyalty_tier -> ...|
|  3|  Cookie|{contains_nuts ->...|
|  4|  Cookie|{contains_nuts ->...|
+---+--------+--------------------+
only showing top 5 rows



Connect to AuraDB

In [ ]:
from graphframes import GraphFrame
from pyspark.sql.functions import col

n = spark.read.format("org.neo4j.spark.DataSource")\
    .option("url", uri)\
    .option("authentication.basic.username", username)\
    .option("authentication.basic.password", password)\
    .option("database","2e66c74e")\
    .option("labels", "Customer") \
    .load() \
    .withColumnRenamed("<id>","id")


e = spark.read.format("org.neo4j.spark.DataSource")\
    .option("url", uri)\
    .option("authentication.basic.username", username)\
    .option("authentication.basic.password", password)\
    .option("database","2e66c74e")\
    .option("relationship", "PURCHASED") \
    .option("relationship.source.labels", "Customer") \
    .option("relationship.target.labels", "Cookie") \
    .load() \
    .withColumnRenamed("<source.id>", "src") \
    .withColumnRenamed("<target.id>", "dst")\

g = GraphFrame(n, e)
g.vertices.show(5)
g.edges.show(5)

+---+----------+--------+------------+-----------+--------------------+
| id|  <labels>|    name|loyalty_tier|home_planet|               email|
+---+----------+--------+------------+-----------+--------------------+
|  0|[Customer]|  Leenew|        Gold|      Earth|      linu@earth.com|
|  1|[Customer]|TaeHyung|      Silver|      Venus|      tatetea@ij.com|
|  2|[Customer]|   JoIly|        Gold|      Titan|jollymonejolly@ab...|
+---+----------+--------+------------+-----------+--------------------+

+-------------------+----------+---+---------------+-----------+-------------------+------------------+--------------------+---+---------------+----------------------+--------------------+--------------------+---------------+-----------------+-------+--------------+
|           <rel.id>|<rel.type>|src|<source.labels>|source.name|source.loyalty_tier|source.home_planet|        source.email|dst|<target.labels>|target.price_per_dozen|  target.flavor_name|target.contains_nuts|target.calories|rel

##6.3. GraphFrames Exploration

In [ ]:
from graphframes import GraphFrame
from pyspark.sql.functions import col
v = df

edges_query = """
MATCH (src)-[r]->(dst)
RETURN id(src) AS src, id(dst) AS dst, type(r) AS edge_label
"""

e = spark.read \
.format("org.neo4j.spark.DataSource") \
.option("url",uri)\
.option("authentication.type", "basic") \
.option("authentication.basic.username", username) \
.option("authentication.basic.password", password) \
.option("query", edges_query) \
.load()

g = GraphFrame(v,e)

print(f"Total Nodes:",{g.vertices.count()})
print(f"Total Edges:",{g.vertices.count()})
print("Unique Node labels:")
g.vertices.select("label").distinct().show(truncate = False)
print("Unique Edge labels:")
g.edges.select("edge_label").distinct().show(truncate = False)

Total Nodes: {9}
Total Edges: {9}
Unique Node labels:
+----------+
|label     |
+----------+
|Ingredient|
|Cookie    |
|Customer  |
+----------+

Unique Edge labels:
+----------+
|edge_label|
+----------+
|CONTAINS  |
|REVIEWED  |
|PURCHASED |
+----------+



### 6.3.1. Find a node based on property and relationship.

In [ ]:
motifs = g.find("(a)-[edge]->(b)")\
          .filter("edge.edge_label = 'CONTAINS' OR edge.edge_label = 'PURCHASED' ")
print("Node connected by CONTAINS or PURCHASED relationships:")
motifs.select("a.label","a.props","edge.edge_label","b.label","b.props").show(1, truncate = False)

print(" Calculating PageRank")
results = g.pageRank(resetProbability=0.15,maxIter=5)

print("\n Most influential nodes (Highest PageRanks):")
results.vertices.orderBy(col("pagerank").desc()).select("label","props","pagerank").show(1,truncate = False)

Node connected by CONTAINS or PURCHASED relationships:
+------+------------------------------------------------------------------------------------------------------+----------+----------+-----------------------------------------------------------+
|label |props                                                                                                 |edge_label|label     |props                                                      |
+------+------------------------------------------------------------------------------------------------------+----------+----------+-----------------------------------------------------------+
|Cookie|{contains_nuts -> true, flavor_name -> Nebula Nutella Sablé, price_per_dozen -> 28.0, calories -> 320}|CONTAINS  |Ingredient|{ingredient_name -> Space Cocoa, allergen_warning -> false}|
+------+------------------------------------------------------------------------------------------------------+----------+----------+------------------------------------

###6.3.2. Find specific nodes based on properties and relationships
To find specific nodes based on properties and relationships, I utilized GraphFrames' structural pattern matching `(g.find("(a)-[e]->(b)")`. By filtering edge_label, I was able  to successfully query and display the exact nodes conencted by `CONTAINS`  and `PURCHASED` relationships directly within spark without writing antive Cypher queries.

###6.3.3 Something interesting!

TO demonstrate graph analytics, I ran the pageRank algorithm over my grapgFrame. It measures the transitive influence of nodes within a network. According to the algorithm, the most influential node in my graph was `Ingredient, {ingredient_name -> Space Cocoa, allergen_warning -> false}`